# Alzheimer's MRI Classification — DenseNet-121 Baseline (Colab A100)

**4-class**: MildDemented · ModerateDemented · NonDemented · VeryMildDemented
**Binary**: AD (Mild + Moderate + VeryMild) vs NonDemented
**Target**: ≥ 90% accuracy on both tasks | Grad-CAM clinical biomarker validation

> Data: Google Drive · Model: DenseNet-121 (ImageNet pretrained) · GPU: A100 · AMP: FP16

In [ ]:
# Mount Google Drive and install grad-cam
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'grad-cam', '-q'])
print("Setup complete.")

In [ ]:
import hashlib, os, time
from collections import Counter
from pathlib import Path

import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from PIL import Image
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
)
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
from torchvision.models import DenseNet121_Weights
from tqdm.notebook import tqdm

# ── Configuration ─────────────────────────────────────────────────
DRIVE_ROOT = "/content/drive/MyDrive"

DATA_CLASSES = {
    "MildDemented":     f"{DRIVE_ROOT}/MildDemented",
    "ModerateDemented": f"{DRIVE_ROOT}/ModerateDemented",
    "NonDemented":      f"{DRIVE_ROOT}/NonDemented",
    "VeryMildDemented": f"{DRIVE_ROOT}/VeryMildDemented",
}

OUTPUT_DIR  = f"{DRIVE_ROOT}/alzheimer_outputs"
CHECKPOINT  = f"{OUTPUT_DIR}/checkpoints/best_model.pth"
GRADCAM_DIR = f"{OUTPUT_DIR}/gradcam"
RESULTS_DIR = f"{OUTPUT_DIR}/results"

for d in [f"{OUTPUT_DIR}/checkpoints", GRADCAM_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

CLASS_ORDER   = ["MildDemented", "ModerateDemented", "NonDemented", "VeryMildDemented"]
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True   # fixed 224×224 input → cuDNN autotuning

# A100-optimised settings
BATCH_SIZE    = 64      # A100 80 GB VRAM handles 64 comfortably
NUM_WORKERS   = 4
USE_AMP       = True    # FP16 mixed precision on A100 (Tensor Cores)

PHASE1_EPOCHS = 10      # frozen features: classifier head warm-up
PHASE2_EPOCHS = 20      # full fine-tune with early stopping
PATIENCE      = 5
MIN_DELTA     = 0.001

PHASE1_LR           = 1e-4
PHASE2_LR_FEATURES  = 1e-5
PHASE2_LR_CLASSIFIER= 1e-4
WEIGHT_DECAY        = 1e-4
N_GRADCAM           = 5

print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU    : {props.name}")
    print(f"VRAM   : {props.total_memory / 1e9:.1f} GB")
    print(f"AMP    : {USE_AMP}")

In [ ]:
# ── Deterministic hash-based split ────────────────────────────────
# Files are UUID-named (no real subject IDs). MD5 hash mod 100
# gives a reproducible 70 / 15 / 15 partition with <0.3% error.

def get_split(uuid_stem: str) -> str:
    h = int(hashlib.md5(uuid_stem.encode("utf-8")).hexdigest(), 16)
    b = h % 100
    return "train" if b < 70 else ("val" if b < 85 else "test")


def build_transforms(split: str) -> transforms.Compose:
    norm = transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    if split == "train":
        return transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=10),
            transforms.ColorJitter(
                brightness=0.2, contrast=0.2, saturation=0.1, hue=0.0
            ),
            transforms.ToTensor(),
            norm,
        ])
    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        norm,
    ])


class AlzheimerDataset(Dataset):
    def __init__(self, split: str):
        self.transform    = build_transforms(split)
        self.class_to_idx = {c: i for i, c in enumerate(CLASS_ORDER)}
        self.samples: list[tuple[str, int]] = []
        for cls_name, cls_dir in DATA_CLASSES.items():
            label = self.class_to_idx[cls_name]
            for p in Path(cls_dir).glob("*.jpg"):
                if get_split(p.stem) == split:
                    self.samples.append((str(p), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        return self.transform(Image.open(path).convert("RGB")), label

    def class_counts(self) -> dict:
        return dict(Counter(CLASS_ORDER[lbl] for _, lbl in self.samples))


def compute_class_weights(train_ds: AlzheimerDataset) -> torch.Tensor:
    counts = train_ds.class_counts()
    total  = sum(counts.values())
    return torch.tensor(
        [total / (len(CLASS_ORDER) * counts[c]) for c in CLASS_ORDER],
        dtype=torch.float32, device=DEVICE,
    )


print("Building datasets (scanning Drive folders)...")
t = time.time()
train_ds = AlzheimerDataset("train")
val_ds   = AlzheimerDataset("val")
test_ds  = AlzheimerDataset("test")
print(f"  Done in {time.time()-t:.1f}s")

ldr_kw = dict(num_workers=NUM_WORKERS, persistent_workers=True, pin_memory=True)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,      shuffle=True,  **ldr_kw)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE * 2,  shuffle=False, **ldr_kw)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE * 2,  shuffle=False, **ldr_kw)

class_weights = compute_class_weights(train_ds)

print(f"\n{'Class':<22} {'Train':>8} {'Val':>8} {'Test':>8}")
print("-" * 50)
for cls in CLASS_ORDER:
    tr = train_ds.class_counts().get(cls, 0)
    vl = val_ds.class_counts().get(cls, 0)
    te = test_ds.class_counts().get(cls, 0)
    print(f"{cls:<22} {tr:>8} {vl:>8} {te:>8}")
print(f"{'TOTAL':<22} {len(train_ds):>8} {len(val_ds):>8} {len(test_ds):>8}")
print(f"\nClass weights (balanced): {[round(w,4) for w in class_weights.tolist()]}")

In [ ]:
# ── DenseNet-121 ──────────────────────────────────────────────────

def build_model() -> nn.Module:
    model = torchvision.models.densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1)
    # Replace 1000-class head with Dropout + 4-class head
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.5),
        nn.Linear(model.classifier.in_features, 4),
    )
    return model.to(DEVICE)


def freeze_features(model: nn.Module) -> None:
    for p in model.features.parameters():
        p.requires_grad = False


def unfreeze_all(model: nn.Module) -> None:
    for p in model.parameters():
        p.requires_grad = True


class EarlyStopping:
    def __init__(self, patience: int, min_delta: float, path: str):
        self.patience  = patience
        self.min_delta = min_delta
        self.path      = path
        self.best_acc  = 0.0
        self.counter   = 0

    def step(self, val_acc: float, model: nn.Module) -> bool:
        if val_acc > self.best_acc + self.min_delta:
            self.best_acc = val_acc
            self.counter  = 0
            torch.save({"model_state_dict": model.state_dict(),
                        "val_acc": val_acc}, self.path)
            return False
        self.counter += 1
        return self.counter >= self.patience

    def reset_counter(self):
        self.counter = 0


model = build_model()
total  = sum(p.numel() for p in model.parameters())
frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f"DenseNet-121 loaded  |  total params: {total:,}")

In [ ]:
# ── Training loop (AMP / mixed precision) ─────────────────────────

def train_one_epoch(model, loader, criterion, optimizer, scaler, epoch, phase):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    pbar = tqdm(loader, desc=f"P{phase}|E{epoch:02d}", leave=False)
    for imgs, labels in pbar:
        imgs   = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast(device_type="cuda", enabled=USE_AMP):
            outputs = model(imgs)
            loss    = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        preds       = outputs.argmax(1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)
        total_loss += loss.item() * labels.size(0)
        pbar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{correct/total:.4f}")
    return total_loss / total, correct / total


@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs   = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        with autocast(device_type="cuda", enabled=USE_AMP):
            outputs = model(imgs)
            loss    = criterion(outputs, labels)
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += labels.size(0)
        total_loss += loss.item() * labels.size(0)
    return total_loss / total, correct / total


def run_phase(phase, model, criterion, optimizer, scheduler, stopper, n_epochs):
    scaler = GradScaler("cuda", enabled=USE_AMP)
    for epoch in range(1, n_epochs + 1):
        tr_loss, tr_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, scaler, epoch, phase)
        vl_loss, vl_acc = validate(model, val_loader, criterion)
        scheduler.step()
        lr = optimizer.param_groups[0]["lr"]
        marker = " ← best" if vl_acc >= stopper.best_acc else ""
        print(f"  P{phase}|E{epoch:02d}  "
              f"train acc={tr_acc:.4f} loss={tr_loss:.4f}  "
              f"val acc={vl_acc:.4f} loss={vl_loss:.4f}  "
              f"lr={lr:.2e}{marker}")
        if stopper.step(vl_acc, model):
            print(f"  Early stopping. Best val acc: {stopper.best_acc:.4f}")
            break

print("Training functions ready.")

In [ ]:
# ── Evaluation & Plots ────────────────────────────────────────────

@torch.no_grad()
def collect_predictions(model, loader):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []
    for imgs, labels in loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        with autocast(device_type="cuda", enabled=USE_AMP):
            outputs = model(imgs)
        probs = torch.softmax(outputs, 1).cpu().numpy()
        preds = outputs.argmax(1).cpu().numpy()
        all_labels.extend(labels.numpy())
        all_preds.extend(preds)
        all_probs.append(probs)
    return all_labels, all_preds, np.vstack(all_probs)


def evaluate_multiclass(model, loader):
    labels, preds, probs = collect_predictions(model, loader)
    report = classification_report(
        labels, preds, target_names=CLASS_ORDER, digits=4, output_dict=True)
    cm = confusion_matrix(labels, preds)
    return report, cm, probs, labels


def evaluate_binary(all_labels, all_probs):
    # AD score = 1 - P(NonDemented). No retraining needed.
    nd_idx    = CLASS_ORDER.index("NonDemented")
    bin_lbl   = (np.array(all_labels) != nd_idx).astype(int)
    ad_score  = 1.0 - all_probs[:, nd_idx]
    bin_pred  = (ad_score >= 0.5).astype(int)
    auc       = roc_auc_score(bin_lbl, ad_score)
    acc       = float((bin_pred == bin_lbl).mean())
    tn, fp, fn, tp = confusion_matrix(bin_lbl, bin_pred).ravel()
    return {
        "accuracy":    acc,
        "auc_roc":     auc,
        "sensitivity": tp / (tp + fn + 1e-9),
        "specificity": tn / (tn + fp + 1e-9),
        "binary_labels": bin_lbl,
        "ad_score":      ad_score,
    }


def plot_confusion_matrix(cm, path):
    short = ["Mild", "Moderate", "Non", "VeryMild"]
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm, cmap="Blues")
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(4)); ax.set_yticks(range(4))
    ax.set_xticklabels(short, rotation=30, ha="right")
    ax.set_yticklabels(short)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title("Confusion Matrix (Test Set)")
    th = cm.max() / 2
    for i in range(4):
        for j in range(4):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > th else "black", fontsize=10)
    plt.tight_layout(); plt.savefig(path, dpi=150); plt.close()
    print(f"  Saved: {path}")


def plot_roc(bin_lbl, ad_score, auc, path):
    fpr, tpr, _ = roc_curve(bin_lbl, ad_score)
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot(fpr, tpr, lw=2, label=f"AUC = {auc:.4f}")
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC Curve — Binary (AD vs NonDemented)")
    ax.legend(loc="lower right")
    plt.tight_layout(); plt.savefig(path, dpi=150); plt.close()
    print(f"  Saved: {path}")

print("Evaluation functions ready.")

In [ ]:
# ── Grad-CAM ──────────────────────────────────────────────────────
# Target layer: model.features.denseblock4
#   → last dense block, output shape [B, 1024, 7, 7]
#   → highest-level semantic features, sufficient spatial resolution

BIOMARKER = {
    "MildDemented":     "Hippocampal atrophy  |  medial temporal lobe",
    "ModerateDemented": "Ventricular enlargement  |  diffuse cortical thinning",
    "NonDemented":      "Normal cortex  |  no focal activation expected",
    "VeryMildDemented": "Subtle hippocampal / entorhinal cortex atrophy",
}


def setup_gradcam(model) -> GradCAM:
    return GradCAM(model=model, target_layers=[model.features.denseblock4])


@torch.no_grad()
def select_samples(model, dataset, n: int = 5) -> dict:
    # Per class: correctly predicted, ranked by confidence (top-n).
    model.eval()
    hits = {i: [] for i in range(4)}
    for idx in tqdm(range(len(dataset)), desc="Selecting Grad-CAM samples", leave=False):
        img, label = dataset[idx]
        out  = model(img.unsqueeze(0).to(DEVICE))
        prob = torch.softmax(out, 1)
        pred = out.argmax(1).item()
        if pred == label:
            hits[label].append((idx, prob[0, pred].item()))
    return {c: [x[0] for x in sorted(v, key=lambda x: x[1], reverse=True)[:n]]
            for c, v in hits.items()}


_raw_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])


def generate_gradcam(model, cam, dataset, selected: dict) -> None:
    model.eval()
    for cls_idx, cls_name in enumerate(CLASS_ORDER):
        idxs = selected.get(cls_idx, [])
        if not idxs:
            print(f"  No samples found for {cls_name}"); continue
        n   = len(idxs)
        fig, axes = plt.subplots(2, n, figsize=(4 * n, 8))
        if n == 1:
            axes = np.array([[axes[0]], [axes[1]]])
        fig.suptitle(
            f"Grad-CAM: {cls_name}\n"
            f"Clinical biomarker: {BIOMARKER[cls_name]}",
            fontsize=11,
        )
        for col, si in enumerate(idxs):
            path, _ = dataset.samples[si]
            raw_img = np.array(
                Image.open(path).convert("RGB").resize((224, 224)),
                dtype=np.float32,
            ) / 255.0
            norm_t, _ = dataset[si]
            gs_cam = cam(
                input_tensor=norm_t.unsqueeze(0).to(DEVICE),
                targets=[ClassifierOutputTarget(cls_idx)],
            )[0]
            overlay = show_cam_on_image(
                raw_img, gs_cam,
                use_rgb=True,
                colormap=cv2.COLORMAP_JET,
                image_weight=0.5,
            )
            axes[0, col].imshow(raw_img)
            axes[0, col].set_title("Original", fontsize=8)
            axes[0, col].axis("off")
            axes[1, col].imshow(overlay)
            axes[1, col].set_title("Grad-CAM", fontsize=8)
            axes[1, col].axis("off")
        plt.tight_layout()
        save_p = f"{GRADCAM_DIR}/gradcam_{cls_name}.png"
        plt.savefig(save_p, dpi=150, bbox_inches="tight")
        plt.close()
        print(f"  Saved: {save_p}")

print("Grad-CAM functions ready.")

In [ ]:
# ── Run Full Pipeline ─────────────────────────────────────────────
t0        = time.time()
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
stopper   = EarlyStopping(patience=PATIENCE, min_delta=MIN_DELTA, path=CHECKPOINT)

# ── Phase 1: frozen features, warm up classifier head ─────────────
print(f"{'='*60}")
print(f"[PHASE 1]  {PHASE1_EPOCHS} epochs  |  features FROZEN")
print(f"{'='*60}")
freeze_features(model)
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {n_trainable:,}  (classifier head only)\n")

opt1 = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=PHASE1_LR, weight_decay=WEIGHT_DECAY,
)
sch1 = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt1, T_max=PHASE1_EPOCHS, eta_min=1e-6)

run_phase(1, model, criterion, opt1, sch1, stopper, PHASE1_EPOCHS)

# ── Phase 2: full fine-tune ───────────────────────────────────────
print(f"\n{'='*60}")
print(f"[PHASE 2]  {PHASE2_EPOCHS} epochs  |  ALL layers unfrozen")
print(f"{'='*60}")
ckpt = torch.load(CHECKPOINT)
model.load_state_dict(ckpt["model_state_dict"])
print(f"Loaded Phase 1 best  (val_acc={ckpt['val_acc']:.4f})\n")

unfreeze_all(model)
stopper.reset_counter()

opt2 = torch.optim.AdamW([
    {"params": model.features.parameters(),   "lr": PHASE2_LR_FEATURES},
    {"params": model.classifier.parameters(), "lr": PHASE2_LR_CLASSIFIER},
], weight_decay=WEIGHT_DECAY)
sch2 = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt2, T_max=PHASE2_EPOCHS, eta_min=1e-7)

run_phase(2, model, criterion, opt2, sch2, stopper, PHASE2_EPOCHS)

# ── Test-set evaluation ───────────────────────────────────────────
print(f"\n{'='*60}")
print("[EVAL]  Test set — loading best checkpoint")
print(f"{'='*60}")
ckpt = torch.load(CHECKPOINT)
model.load_state_dict(ckpt["model_state_dict"])
print(f"Best checkpoint val_acc={ckpt['val_acc']:.4f}\n")

report, cm, all_probs, all_labels = evaluate_multiclass(model, test_loader)
binary = evaluate_binary(all_labels, all_probs)

print("--- 4-Class Classification Report ---")
for cls in CLASS_ORDER:
    m = report[cls]
    print(f"  {cls:<22}  "
          f"P={m['precision']:.4f}  R={m['recall']:.4f}  F1={m['f1-score']:.4f}  "
          f"N={int(m['support'])}")
print(f"  {'Overall Accuracy':<22}  {report['accuracy']:.4f}")
print(f"  {'Macro F1':<22}  {report['macro avg']['f1-score']:.4f}")

print("\n--- Binary (AD vs NonDemented) ---")
print(f"  Accuracy    : {binary['accuracy']:.4f}")
print(f"  AUC-ROC     : {binary['auc_roc']:.4f}")
print(f"  Sensitivity : {binary['sensitivity']:.4f}  (AD detection rate)")
print(f"  Specificity : {binary['specificity']:.4f}  (Normal exclusion rate)")

# ── Save plots ────────────────────────────────────────────────────
plot_confusion_matrix(cm, f"{RESULTS_DIR}/confusion_matrix.png")
plot_roc(binary["binary_labels"], binary["ad_score"],
         binary["auc_roc"], f"{RESULTS_DIR}/roc_curve.png")

# Show inline
from IPython.display import Image as IPImage, display
display(IPImage(f"{RESULTS_DIR}/confusion_matrix.png"))
display(IPImage(f"{RESULTS_DIR}/roc_curve.png"))

# ── Write text report ─────────────────────────────────────────────
rp = f"{RESULTS_DIR}/metrics_report.txt"
with open(rp, "w") as f:
    f.write("=== 4-Class Classification Report ===\n")
    for cls in CLASS_ORDER:
        m = report[cls]
        f.write(f"{cls}: P={m['precision']:.4f} R={m['recall']:.4f} "
                f"F1={m['f1-score']:.4f} N={int(m['support'])}\n")
    f.write(f"Overall Accuracy : {report['accuracy']:.4f}\n")
    f.write(f"Macro F1         : {report['macro avg']['f1-score']:.4f}\n\n")
    f.write("=== Binary (AD vs NonDemented) ===\n")
    f.write(f"Accuracy    : {binary['accuracy']:.4f}\n")
    f.write(f"AUC-ROC     : {binary['auc_roc']:.4f}\n")
    f.write(f"Sensitivity : {binary['sensitivity']:.4f}\n")
    f.write(f"Specificity : {binary['specificity']:.4f}\n")
print(f"Metrics saved: {rp}")

# ── Grad-CAM ──────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("[GRAD-CAM]  Generating visualizations")
print(f"{'='*60}")
cam      = setup_gradcam(model)
selected = select_samples(model, test_ds, n=N_GRADCAM)
generate_gradcam(model, cam, test_ds, selected)

for cls_name in CLASS_ORDER:
    p = f"{GRADCAM_DIR}/gradcam_{cls_name}.png"
    if os.path.exists(p):
        display(IPImage(p))

# ── Summary ───────────────────────────────────────────────────────
elapsed = (time.time() - t0) / 60
print(f"\n{'='*60}")
print("PIPELINE COMPLETE")
print(f"{'='*60}")
print(f"  4-class accuracy  : {report['accuracy']:.4f}")
print(f"  Binary accuracy   : {binary['accuracy']:.4f}")
print(f"  Binary AUC-ROC    : {binary['auc_roc']:.4f}")
print(f"  Total time        : {elapsed:.1f} min")
print(f"  Best model        : {CHECKPOINT}")
print(f"  Outputs           : {OUTPUT_DIR}")